In [ ]:
!pip -q install diffusers
print("Done")

Done


setup -Load the local language model

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_NAME).to(device)
llm_model.eval()
print("Language model ready.")

Running on: cuda


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Language model ready.


In [ ]:
import torch

def chat(messages, temperature=0.7, max_tokens=200):
    """Send a list of {"role", "content"} messages to the local model and return its reply."""
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(text, return_tensors="pt").to(device)

    gen_args = dict(max_new_tokens=max_tokens, pad_token_id=llm_tokenizer.eos_token_id)
    if temperature > 0:
        gen_args.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        gen_args.update(do_sample=False)

    with torch.no_grad():
        output = llm_model.generate(**inputs, **gen_args)

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def ask(prompt, system=None, temperature=0.7, max_tokens=200):
    """One prompt in, one reply out. Used by most programs below."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return chat(messages, temperature, max_tokens)

In [ ]:
import re

def build_prompt(role, task, fmt, constraints=""):
    """Role / Task / Format / Constraints template (Program 2 explains it)."""
    return f"""Role: {role}
Task: {task}
Format: {fmt}
Constraints: {constraints}"""

def extract_code(reply):
    """Pull the Python code out of a model reply, with or without  fences."""
    match = re.search(r"(?:python)?\s*\n(.*?)```", reply, re.DOTALL)
    return (match.group(1) if match else reply).strip()

def run_generated_code(code):
    """Run model-written code safely enough for a lab: read it first, and catch errors."""
    try:
        exec(code, globals())
        return True
    except Exception as e:
        print(f"The model's code failed: {type(e)._name_}: {e}")
        print("That is a real limitation of small models. Note it in your report.")
        return False

print("Helpers ready.")

Helpers ready.


In [ ]:
print(ask("Say hello to all bsc Hons Data Science students in one short sentance.",max_tokens=40))

Hello all BSc Hons Data Science students! I hope you're having an excellent year and thriving with your studies.


In [ ]:
vague_prompt = "Write something about food."

structured_prompt = """
Role: You are a friendly nutrition blogger writing for college
students.
Task: Write a 3-sentence post about the benefits of eating
breakfast.
Format: Casual tone, no more than 60 words.
Constraint: Do not mention any specific brand names.
"""

print("----- VAGUE PROMPT OUTPUT -----")
print(ask(vague_prompt))
print("\n----- STRUCTURED PROMPT OUTPUT -----")
print(ask(structured_prompt))


----- VAGUE PROMPT OUTPUT -----
Food is an essential part of human life and culture, playing a crucial role in satisfying our basic needs for nutrition and pleasure. It can be prepared in countless ways using various ingredients and cooking techniques to create a wide range of dishes that reflect the diversity of cultures around the world.

The act of preparing and eating food has also been an important social activity throughout history, with meals often serving as opportunities to gather families and friends together or to celebrate special occasions such as holidays and weddings. Many different types of cuisine exist across the globe, each with its own unique flavors, textures, and cultural traditions.

In addition to being enjoyed as sustenance, food can also have symbolic meaning in many cultures, with certain foods having specific associations or connotations that are believed to bring good fortune, health, or protection against evil spirits. For example, rice is considered a sym

In [ ]:
scenarios = [
    ("a polite customer support agent", "apologise for a late delivery and offer a discount code", "a short email, under 80 words"),
    ("an enthusiastic sports coach", "motivate a team before a final match", "a 4-line pep talk"),
]

for role, task, fmt in scenarios:
    prompt = build_prompt(role, task, fmt)
    print(f"=== {task} ===")
    print(ask(prompt))
    print()

=== apologise for a late delivery and offer a discount code ===
Subject: Sorry for the Delays - Thank You for Your Patience

Dear [Customer's Name],

I apologize for any inconvenience caused with my recent shipment. My team is currently working diligently to expedite this order. In recognition of your patience during these delays, I would like to offer you a 10% discount on your next purchase.

Thank you for understanding and please let me know if there’s anything else I can assist you with.

Best regards,

[Your Full Name]  
[Your Position]  
[Company Name]  
[Contact Information]

=== motivate a team before a final match ===
Here's a 4-line pep talk that could be delivered to a sports team before a final match:

"Believe in yourselves! Your skills and heart set you apart.
With each score, your passion ignites the crowd anew.
Let's sprint forward, past the finish line so bright.
Your victory is within reach, let's make it real!"

